# True QLoRA: Training Apple's 3B Model with 4-bit NF4

Uses bitsandbytes NF4 quantization on the frozen base model.
Only ~5GB GPU memory — fits on free T4 with massive headroom.

This is proper QLoRA as defined by [Dettmers et al. 2023](https://arxiv.org/abs/2305.14314):
4-bit quantized base + fp32 LoRA adapters.

## 1. Setup

Upload to `My Drive/hunch-training/`:
- `adapter_training_toolkit_v26_0_0/` (from developer.apple.com)
- `prepare_data.py`, `train_qlora_full.py`, `tldr_bank.db`, `prompts.jsonl`

In [1]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/hunch-training'
WORK_DIR = '/content/hunch-training'

!mkdir -p {WORK_DIR}
!cp -r {DRIVE_DIR}/adapter_training_toolkit_v26_0_0 {WORK_DIR}/
!cp {DRIVE_DIR}/prepare_data.py {WORK_DIR}/
!cp {DRIVE_DIR}/train_qlora_full.py {WORK_DIR}/
!mkdir -p {WORK_DIR}/../bank {WORK_DIR}/../benchmark
!cp {DRIVE_DIR}/tldr_bank.db {WORK_DIR}/../bank/
!cp {DRIVE_DIR}/prompts.jsonl {WORK_DIR}/../benchmark/

!cd {WORK_DIR}/adapter_training_toolkit_v26_0_0 && pip install -r requirements.txt -q
!pip install bitsandbytes -q

import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 62.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.6/362.6 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.0 MB/s eta 0:00:00:00:0100:01
CUDA: True
GPU: Tesla T4


In [10]:
!cp {DRIVE_DIR}/prepare_data.py {WORK_DIR}/

## 2. Prepare training data

In [11]:
!cd {WORK_DIR} && python3 prepare_data.py --sources override

Loaded 21478 entries from bank
Filtered to sources {'override'}: 134 entries (from 21478)
  override: 134
Excluded 38 entries matching benchmark prompts
After dedup: 96 unique entries (removed 0)
Small dataset — using all 96 examples for both train and eval
Wrote 96 examples to /content/hunch-training/train.jsonl
Wrote 96 examples to /content/hunch-training/eval.jsonl

Sample training examples:
  user: show response headers
  asst: curl -I https://example.com

  user: dns lookup for a domain
  asst: dig example.com

  user: record shell session to file
  asst: script session.log



## 3. Train

No patches needed — `train_qlora_full.py` handles everything:
mmap loading, NF4 quantization, training loop.

~5GB GPU memory. Can use large batch sizes on T4.

In [ ]:
!cd {WORK_DIR} && python3 train_qlora_full.py \
  --epochs 20 \
  --batch-size 8 \
  --learning-rate 1e-4 \
  --checkpoint-dir qlora-override-checkpoints/

Device: cuda | RAM=0.6GB GPU=0.0GB
Quantized 280 layers to NF4
Trainable: 67M params | RAM=6.3GB GPU=2.2GB
Train: 96 examples, 12 batches
Eval: 96 examples

Training: 20 epochs, batch 8, lr 0.0001

Epoch 1/20
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:2954: UserWarning: Mismatch dtype between input and weight: input dtype = float, weight dtype = c10::Half, Cannot dispatch to fused implementation. (Triggered internally at /pytorch/aten/src/ATen/native/layer_norm.cpp:344.)
  return torch.rms_norm(input, normalized_shape, weight, eps)
Saved 700 adapter weights (254MB) to qlora-override-checkpoints//adapter-epoch1.pt
  Train loss: 3.4408 | Eval loss: 1.6959 | RAM=1.5GB GPU=3.0GB

Epoch 2/20
Saved 700 adapter weights (254MB) to qlora-override-checkpoints//adapter-epoch2.pt
  Train loss: 0.8634 | Eval loss: 0.4109 | RAM=1.5GB GPU=3.0GB

Epoch 3/20
Saved 700 adapter weights (254MB) to qlora-override-checkpoints//adapter-epoch3.pt
  Train loss: 0.3740 | Eval loss: 0.2714 | 

## 4. Save checkpoints

In [13]:
!cp -r {WORK_DIR}/qlora-override-checkpoints/ {DRIVE_DIR}/qlora-override-checkpoints
!echo 'Checkpoints saved to Drive'

Checkpoints saved to Drive


## 5. Evaluate

In [9]:
import json, subprocess

test_prompts = [
    'find files changed in the last hour',
    'show disk usage',
    'generate a random password',
    'kill a process by name',
    'show http headers of a url',
    'record terminal session',
    'find files larger than 100mb',
    'convert image to different format',
    'show all listening ports',
    'find files modified in the last 7 days',
    'find files owned by root',
    'count lines in all python files',
    'show all environment variables',
    'clear the terminal',
    'compare two files',
]

system = 'Output a single shell command for zsh on macOS. No explanation, no markdown, no backticks. Just the command.'

with open(f'{WORK_DIR}/test_prompts.jsonl', 'w') as f:
    for p in test_prompts:
        f.write(json.dumps([
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': p}
        ]) + '\n')

result = subprocess.run(
    ['python3', '-m', 'examples.generate',
     '--prompt', '../test_prompts.jsonl',
     '--checkpoint', '../qlora-checkpoints/adapter-final.pt',
     '--precision', 'f16-mixed'],
    capture_output=True, text=True,
    cwd=f'{WORK_DIR}/adapter_training_toolkit_v26_0_0'
)

lines = (result.stdout + result.stderr).strip().split('\n')
idx = 0
for line in lines:
    if 'Response for prompt' in line:
        answer = line.split(': ', 2)[-1].replace('<turn_end>', '').strip()
        prompt = test_prompts[idx] if idx < len(test_prompts) else '?'
        print(f'Q: {prompt:<45} A: {answer}')
        idx += 1

if idx == 0:
    print('No output. Check error:')
    print('STDERR:', result.stderr[-500:])
    print('Return code:', result.returncode)

No output. Check error:
STDERR: 
Return code: -9


## 6. Export .fmadapter

In [14]:
!cd {WORK_DIR}/adapter_training_toolkit_v26_0_0 && python3 -m export.export_fmadapter \
  --adapter-name hunch_qlora \
  --checkpoint ../qlora-override-checkpoints/adapter-final.pt \
  --output-dir ../qlora-override-exports/

!ls -lh {WORK_DIR}/qlora-override-exports/
!cp -r {WORK_DIR}/qlora-override-exports {DRIVE_DIR}/qlora-override-exports
!echo 'Adapter exported and saved to Drive'

scikit-learn version 1.6.1 is not supported. Minimum required version: 0.17. Maximum required version: 1.5.1. Disabling scikit-learn conversion API.
XGBoost version 3.2.0 has not been tested with coremltools. You may run into unexpected errors. XGBoost 1.4.2 is the most recent version that has been tested.
2026-04-15 16:39:35.262475: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776271175.518226   51164 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776271175.586162   51164 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776271176.102805   51164 computation_placer.cc:177] computation placer already registered. Please check linkage and 

In [8]:
!ls -lh {WORK_DIR}/qlora-exports/
!cp -r {WORK_DIR}/qlora-exports {DRIVE_DIR}/qlora-exports2
!echo 'Adapter exported and saved to Drive'

total 4.0K
drwxr-xr-x 2 root root 4.0K Apr 15 15:49 hunch_qlora.fmadapter
Adapter exported and saved to Drive
